# Referencia- és kiterjesztett implementáció konzisztencia-ellenőrzése

Ez a notebook a **„A Crouch-féle szimulációs prevalencia-becslés kiterjesztése folytonos időbeli modellezésre”** című szakdolgozat validációs anyaga.  
A dolgozat a Crouch-féle incidencia–túlélés alapú Monte Carlo prevalenciabecslés elméleti keretére és a meglévő `rprev` referencia-implementációra épül. A központi feladat a referencia-eljárás több indexdátum együttes kezelésére szolgáló kiterjesztése, valamint ennek módszertani és numerikus validálása (lásd: 3–5. fejezet).

Ennek megfelelően a **referencia** implementáció definíció szerint egyetlen indexdátumhoz ad prevalencia-becslést, ezért több indexdátum esetén az eredmény egyindexű `prevalence()` hívások sorozatából állítható elő. A **kiterjesztett implementáció** ezzel szemben több indexdátum együttes kezelésére készült, és a teljes indexrácsra konzisztens becslést ad egy közös futási keretben (lásd: 4. fejezet – referencia implementáció és a kiterjesztés tárgya; 5. fejezet – matematikai modell és csomagszintű illesztés).

## Validációs cél

A notebook célja annak vizsgálata, hogy a referencia implementációból indexdátumonként összeállított kimenet és a natív többindexű kiterjesztett futtatás kimenete azonos bemenetek (adat, modellek), azonos indexrács és azonos további paraméterek mellett összevethető-e, továbbá hogy a megfigyelt eltérések a kiterjesztés definiált módszertani és algoritmikus sajátosságai mellett értelmezhetők-e.

## Egyezési kritériumok és eltérésértelmezés

A két implementáció futási szerkezete eltér, ezért a véletlenszám-generátor hívásainak száma és ütemezése sem azonos (lásd dolgozat 5. fejezet). A referencia-megoldás indexdátumonként külön futtatással állít elő prevalencia-becslést, így a bináris státuszok véletlen komponense is indexdátumonként, elkülönülő Bernoulli-sorsolásokon keresztül képződik. Ezzel szemben a kiterjesztett megoldásban a státuszképzés véletlen komponense esetszinten egyetlen `xi ~ Unif(0,1)` küszöbhöz kötött, és ezt hasonlítjuk össze az indexrács pontjaihoz tartozó túlélési valószínűségekkel. A két státuszképzési szervezés különbsége miatt azonos induló véletlenmag mellett sem várható bitazonos kimenet; az összevetést a kimeneti összefoglalók (pontbecslés, intervallumhatárok) szintjén végezzük.

A referencia és a kiterjesztett implementáció kimenetét akkor tekintjük konzisztensnek, ha a relatív eltérések az indexrácson kontrolláltak. A pontbecslésre a

$$
\delta_{\widehat{P}}(t_k)
=
\frac{\widehat{P}_{\mathrm{ext}}(t_k)-\widehat{P}_{\mathrm{ref}}(t_k)}{\widehat{P}_{\mathrm{ref}}(t_k)}
$$

relatív eltérést, míg az intervallumhatárokra (minden $\alpha\in\{0.025,\,0.975\}$ esetén) a

$$
\delta_{q,\alpha}(t_k)
=
\frac{q_{\alpha,\mathrm{ext}}(t_k)-q_{\alpha,\mathrm{ref}}(t_k)}{q_{\alpha,\mathrm{ref}}(t_k)}
$$

relatív eltérést képezzük. Az egyezést elsődlegesen a rácson vett tipikus eltérés nagyságrendje alapján értékeljük: rögzített $N_{\mathrm{boot}}=1000$ (azonosan $N_{\mathrm{sim}}=1000$) mellett akkor tekintjük teljesültnek, ha a $|\delta_{\widehat{P}}(t_k)|$ és a $|\delta_{q,\alpha}(t_k)|$ eloszlásának $0.99$-kvantilise nem haladja meg az $\varepsilon=10^{-3}$ küszöböt. A maximális eltéréseket külön riportáljuk, és azok alapján ellenőrizzük, hogy a kilógó pontok nem mutatnak a teljes indexrácsot érintő, tartós mintázatot.

A konzisztencia értelmezéséhez két diagnosztikai ellenőrzést is végzünk:

- az eltérések az indexrács mentén nem mutatnak tartós, egyirányú eltolódást;

- a szimulációs ismétlésszám növelésével ($N_{\mathrm{sim}}$, illetve ahol releváns $N_{\mathrm{boot}}$) a relatív eltérések a nullához konvergálnak, vagyis az indexrácson mért $|\delta_{\widehat{P}}(t_k)|$ és $|\delta_{q,\alpha}(t_k)|$ mennyiségek rendre csökkennek.

## Notebook felépítése

1. Közös paraméterezés rögzítése.  
2. Referencia-kimenet előállítása indexdátumonkénti, egyindexű `prevalence()` hívások sorozatával.  
3. Kiterjesztett implementáció futtatása a teljes indexrácsra.  
4. Célzott összevetés a kimenetszintű mennyiségekre (pontbecslés és intervallumhatárok) az indexrácson.  
5. Konvergenciaellenőrzés: a szimulációs ismétlésszám növelésével ($N_{\mathrm{sim}}$, illetve paraméterben $N_{\mathrm{boot}}$) a referencia--kiterjesztett relatív eltérések nullához tartásának vizsgálata.

---
### 1. Közös paraméter konfiguráció
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [1]:
# Common configuration (shared across runs)
seeds <- c(20260302, 20260303, 20260304)
seed <- seeds[1]
N_boot <- 1000L
num_years <- c(3, 5, 10, 13, 15, 20, 25)

# Index date grid (K = number of index dates)
K <- 3L
index_start <- as.Date("2012-01-07")
index_dates <- seq.Date(from = index_start, by = "year", length.out = K)

# Model + prevalence settings
inc_formula <- entrydate ~ sex
surv_formula <- survival::Surv(time, status) ~ age + sex
dist <- "weibull"
population_size <- 1e6
death_column <- "eventdate"
status_col <- "status"


### 2. Környezet előkészítése
- Betöltjük a szükséges csomagokat (`rprev`, `survival`, `lubridate`), valamint a `prevsim` példaadatot, hogy a referenciafuttatások egységes környezetben és bemeneten induljanak.
- A betöltött adatokból előállítjuk a futtatásokhoz használt munkatáblát (`base_df`), amelyet a későbbi összevetések során konzisztensen használunk.

In [2]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
  library(lubridate)
})

# Példaadat betöltése (kiinduló tábla)
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)


Warning message:
"package 'rprev' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"


### 3. Referencia `rprev` implementáció futtatása
- A referencia implementáció `prevalence()` függvénye egyetlen indexdátumra ad becslést. Több indexdátum esetén a viszonyítási eredmény indexdátumonként ismételt különálló futtatásokból áll elő.
- A kiértékelést több seed beállítás mellett is elvégezzük.

In [3]:
# K>1 (több indexdátum, ismételt K=1 futások)
registry_start_ref <- min(prevsim$entrydate, na.rm = TRUE)

ref_abs_table_by_seed <- list()

# Gyors áttekintés: abszolút prevalencia indexdátumonként, horizontonként
extract_abs <- function(res) {
  out <- sapply(num_years, function(y) {
    as.numeric(res$estimates[[paste0("y", y)]][["absolute.prevalence"]])
  })
  names(out) <- paste0("y", num_years)
  out
}

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  # Reprodukálható viszonyítás: az indexdátumonkénti referencia-futtatásokat ugyanazzal a seeddel indítjuk,
  # így a K>1 viszonyítási sorozat indexek közötti különbségeit nem a véletlenszám-hívások "elmászó" állapota,
  # hanem az eltérő indexdátumokhoz tartozó becslési helyzet magyarázza.
  res_ref_by_index_s <- lapply(index_dates, function(idx) {
    set.seed(s)
    rprev::prevalence(
      index = idx,
      num_years_to_estimate = num_years,
      data = base_df,
      inc_formula = inc_formula,
      surv_formula = surv_formula,
      dist = dist,
      population_size = population_size,
      death_column = death_column,
      registry_start_date = registry_start_ref,
      N_boot = N_boot
    )
  })
  names(res_ref_by_index_s) <- as.character(index_dates)

  ref_abs_table_s <- t(sapply(res_ref_by_index_s, extract_abs))
  ref_abs_table_by_seed[[as.character(s)]] <- ref_abs_table_s
  print(ref_abs_table_s)

  if (s == seeds[1]) {
    res_ref_by_index <- res_ref_by_index_s
    ref_abs_table <- ref_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ref_abs_table



--------------------------------------------------------------------------------
seed = 20260302 
            y3  y5    y10   y13    y15    y20    y25
2012-01-07 207 297 508.49 598.4 650.73 762.22 850.62
2013-01-07 207 304 517.00 606.0 658.32 769.98 858.73
2014-01-07 127 226 433.00 530.3 582.69 694.12 782.52

--------------------------------------------------------------------------------
seed = 20260303 
            y3  y5    y10    y13    y15    y20    y25
2012-01-07 207 297 508.26 597.34 650.02 762.02 851.34
2013-01-07 207 304 517.00 606.35 658.66 770.77 860.07
2014-01-07 127 226 433.00 529.76 582.51 694.44 783.76

--------------------------------------------------------------------------------
seed = 20260304 
            y3  y5   y10    y13    y15    y20    y25
2012-01-07 207 297 508.1 597.32 650.09 761.20 849.95
2013-01-07 207 304 517.0 606.36 658.96 770.27 859.16
2014-01-07 127 226 433.0 529.83 582.68 693.70 782.45


### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja.

In [4]:
# Projekt lokális gyökérkönyvtárának meghatározása (git repo root)
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# Telepített rprev leválasztása, majd lokális forrásból betöltés
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

# Betöltött csomag elérési útjának megjelenítése
cat(
  "Loaded rprev from: ",
  # normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  # "\n",
  sep = ""
)

Warning message:
"package 'testthat' was built under R version 4.4.3"


Loaded rprev from: 

In [5]:
# Git meta rögzítése: futtatott kódverzió azonosítása
git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) return(NA_character_)
  trimws(out[[1]])
}

old_wd <- getwd()                         # munkakönyvtár mentése
on.exit(setwd(old_wd), add = TRUE)        # visszaállítás a cella végén
setwd(project_root)                       # git parancsok futtatása a repo gyökeréből

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/ref-ext-consistency
Git commit: 47f7c7386b7642dd886617933f4e378256e7b46c


### 5. Kiterjesztett `rprev` implementáció futtatása
- Az előző blokkban betöltött lokális implementáció `prevalence()` függvényének futtatása ugyanazzal az indexdátum- és paraméterkészlettel, mint a referenciafuttatásoknál.
- A kiértékelést több seed beállítás mellett is elvégezzük.

In [6]:
years_lbl <- paste0("y", num_years)

ext_abs_table_by_seed <- list()

# A seedek listáját a közös konfigurációban adjuk meg: `seeds`
for (s in seeds) {
  cat("\n", paste(rep("-", 80), collapse = ""), "\n", sep = "")
  cat("seed =", s, "\n")

  set.seed(s)
  res_ext_kmulti_s <- prevalence(
    index = index_dates,
    num_years_to_estimate = num_years,
    data = base_df,
    inc_formula = inc_formula,
    surv_formula = surv_formula,
    dist = dist,
    population_size = population_size,
    death_column = death_column,
    registry_start_date = registry_start_ext,
    N_boot = N_boot
  )
  print(summary(res_ext_kmulti_s))

  ext_abs_table_s <- do.call(
    cbind,
    lapply(num_years, function(y) {
      est <- res_ext_kmulti_s$estimates[[paste0("y", y)]]
      est <- est[order(est$index_date), , drop = FALSE]
      as.numeric(est[["absolute.prevalence"]])
    })
  )
  colnames(ext_abs_table_s) <- years_lbl
  rownames(ext_abs_table_s) <- as.character(sort(unique(res_ext_kmulti_s$index_dates)))
  ext_abs_table_by_seed[[as.character(s)]] <- ext_abs_table_s
  print(ext_abs_table_s)

  if (s == seeds[1]) {
    res_ext_kmulti <- res_ext_kmulti_s
    ext_abs_table <- ext_abs_table_s
  }
}

# A további összevetésekhez a `seeds[1]` futtatás eredményeit használjuk
# ext_abs_table


--------------------------------------------------------------------------------
seed = 20260302 


ERROR: Error: object 'registry_start_ext' not found


### 6. Eredmények összehasonlítása (referencia vs kiterjesztett)
- Az alábbi táblázat az abszolút prevalencia-becsléseket veti össze indexdátumonként és időhorizontonként.
- A `counted` oszlop a regiszterből közvetlenül számlált prevalens esetszámot mutatja (indexdátumonként), a `ref` és `ext` oszlopok pedig az eljárások becsléseit.
- A `delta` és a `delta_pct` oszlopok a referencia-megoldást tekintik bázisnak: `delta = kiterjesztett - referencia`, `delta_pct = 100 * delta / referencia`.


In [ ]:
# Dinamikus összevetés: referencia (ismételt K=1) vs kiterjesztett (natív többindexű), seed szerint

counted_by_date <- sapply(res_ref_by_index, function(res) as.numeric(res$counted))
names(counted_by_date) <- names(res_ref_by_index)

build_compare_df <- function(ref_mat, ext_mat) {
  idx <- intersect(rownames(ref_mat), rownames(ext_mat))
  yrs <- intersect(colnames(ref_mat), colnames(ext_mat))

  df <- expand.grid(index_date = idx, horizon = yrs, stringsAsFactors = FALSE)
  df$ref <- mapply(function(r, c) as.numeric(ref_mat[r, c]), df$index_date, df$horizon)
  df$ext <- mapply(function(r, c) as.numeric(ext_mat[r, c]), df$index_date, df$horizon)

  counted_i <- counted_by_date[as.character(idx)]
  df$counted <- as.numeric(counted_i[df$index_date])

  df$delta <- df$ext - df$ref
  df$delta_pct <- ifelse(df$ref == 0, NA_real_, 100 * df$delta / df$ref)

  df$horizon_years <- as.integer(sub("^y", "", df$horizon))
  df <- df[order(as.Date(df$index_date), df$horizon_years), ]
  df$horizon <- NULL
  df <- df[, c("index_date", "horizon_years", "counted", "ref", "ext", "delta", "delta_pct")]

  df$ref <- round(df$ref, 2)
  df$ext <- round(df$ext, 2)
  df$delta <- round(df$delta, 2)
  df$delta_pct <- round(df$delta_pct, 2)

  df
}

compare_list <- lapply(seeds, function(s) {
  ref_mat <- ref_abs_table_by_seed[[as.character(s)]]
  ext_mat <- ext_abs_table_by_seed[[as.character(s)]]
  df <- build_compare_df(ref_mat, ext_mat)
  df$seed <- s
  df
})

compare_all <- data.table::rbindlist(compare_list, fill = TRUE)
compare_all <- compare_all[order(compare_all$seed, as.Date(compare_all$index_date), compare_all$horizon_years), ]
compare_all <- compare_all[, c("seed", "index_date", "horizon_years", "counted", "ref", "ext", "delta", "delta_pct")]

compare_all


seed,index_date,horizon_years,counted,ref,ext,delta,delta_pct
<dbl>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
20260302,2012-01-07,3,475,207.00,207.00,0.00,0.00
20260302,2012-01-07,5,475,297.00,297.00,0.00,0.00
20260302,2012-01-07,10,475,508.49,508.34,-0.15,-0.03
20260302,2012-01-07,13,475,598.40,597.94,-0.46,-0.08
20260302,2012-01-07,15,475,650.73,650.40,-0.33,-0.05
20260302,2012-01-07,20,475,762.22,762.33,0.11,0.01
20260302,2012-01-07,25,475,850.62,851.55,0.93,0.11
20260302,2013-01-07,3,517,207.00,207.00,0.00,0.00
20260302,2013-01-07,5,517,304.00,304.00,0.00,0.00


#### 7. Különbségek összefoglalása

Az alábbi összesítő táblázat a seedek mentén előállított összehasonlító táblák (`compare_all`) alapján, indexdátumonként és horizontonként az abszolút eltérés (`|delta|`) leíró statisztikáit adja meg.

- A sorok egy-egy indexdátum–horizont párt jelentenek.
- Az oszlopok a seed-választásból adódó ingadozás mértékét írják le (`min`, `max`, `median`, `mean`, `sd`).

Ez a táblázat közvetlen viszonyítási alapot ad annak megítéléséhez, hogy a referencia és a kiterjesztett megoldás közötti eltérés nagyságrendje mennyiben illeszkedik a seed-függő változékonyság tartományába.


In [ ]:
# Összefoglalás: |delta| leíró statisztikák seedek mentén

cmp <- data.table::as.data.table(compare_all)
cmp[, abs_delta := abs(delta)]

summary_abs_delta <- cmp[
  , .(
    ref = {
      v <- ref[seed == seeds[1]]
      if (length(v) == 0) v <- ref
      as.numeric(v[1])
    },
    abs_delta_min = min(abs_delta, na.rm = TRUE),
    abs_delta_max = max(abs_delta, na.rm = TRUE),
    abs_delta_med = stats::median(abs_delta, na.rm = TRUE),
    abs_delta_mean = mean(abs_delta, na.rm = TRUE),
    abs_delta_sd = stats::sd(abs_delta, na.rm = TRUE)
  ),
  by = .(index_date, horizon_years)
]

summary_abs_delta <- summary_abs_delta[order(as.Date(index_date), horizon_years)]
summary_abs_delta[, `:=`(
  ref = round(ref, 2),
  abs_delta_min = round(abs_delta_min, 2),
  abs_delta_max = round(abs_delta_max, 2),
  abs_delta_med = round(abs_delta_med, 2),
  abs_delta_mean = round(abs_delta_mean, 2),
  abs_delta_sd = round(abs_delta_sd, 2)
)]

summary_abs_delta


index_date,horizon_years,ref,abs_delta_min,abs_delta_max,abs_delta_med,abs_delta_mean,abs_delta_sd
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2012-01-07,3,207.00,0.00,0.00,0.00,0.00,0.00
2012-01-07,5,297.00,0.00,0.00,0.00,0.00,0.00
2012-01-07,10,508.49,0.04,0.15,0.05,0.08,0.06
2012-01-07,13,598.40,0.10,0.56,0.46,0.37,0.24
2012-01-07,15,650.73,0.25,0.54,0.33,0.37,0.15
2012-01-07,20,762.22,0.10,1.64,0.11,0.62,0.89
2012-01-07,25,850.62,0.19,2.53,0.93,1.22,1.20
2013-01-07,3,207.00,0.00,0.00,0.00,0.00,0.00
2013-01-07,5,304.00,0.00,0.00,0.00,0.00,0.00


### 10. Megállapítás

A fenti prefix-tábla oszlopaiban ugyanazt az eljárást futtatjuk eltérő horizontkészletekkel (a `num_years` vektor prefixeivel), miközben minden futtatás előtt ugyanazt a véletlenszám-generátor magot (seedet) állítjuk be. Ennek ellenére a közös horizontokra kapott becslések nem feltétlenül egyeznek, mert a horizontkészlet megváltoztatása a szimuláció teljes lefutását is megváltoztatja.

A fő ok a véletlenszám-felhasználás eltérő ütemezése és mennyisége:

- A `prevalence()` a szimuláció induló dátumát a **legnagyobb kért horizont** alapján állítja be: `sim_start_date <- min(index_dates) - years(max(num_years_to_estimate))` (R/prevalence.R). Ha a horizontlistát bővítjük, a `sim_start_date` korábbra kerül.
- A `sim_prevalence()` ezután a szimulációs ablak hosszát a `starting_date` és a legkésőbbi indexdátum különbségéből számolja: `number_incident_days <- difftime(max(index_dates), starting_date, ...)` (R/prevalence.R). A hosszabb ablak **más (és tipikusan több) szimulált incidens esetet** eredményezhet a `draw_incident_population(...)` hívásban, ami a véletlenszám-generátor állapotát is másképp "fogyasztja".
- A kiterjesztett Monte Carlo magban futásonként egyszer sorsolunk esetszintű küszöböt: `xi <- runif(nrow(incident_population))`, majd ezt az összes indexdátumhoz felhasználjuk: `alive_k <- I(xi <= p_k)` (R/prevalence.R). Ha a szimulált incidens populáció mérete (és így a `xi` hossza) megváltozik, akkor ugyanazon kezdő seed mellett is eltérő RNG-állapotból folytatódnak a további lépések.

A referencia megoldáshoz képest a kiterjesztés a státuszképzés véletlen komponensét is másképp szervezi: a túlélési valószínűségekhez tartozó bináris státusz nem független Bernoulli-sorsolások sorozataként áll elő, hanem az esetszintű `xi ~ Unif(0,1)` küszöb és a `p_k` túlélési valószínűségek összehasonlításával. Ez azonos seed használat mellett is megváltoztatja a véletlenszám-hívások mintázatát, ezért ugyanazon véletlenmag mellett sem várható szigorú numerikus egyezés.

Módszertani szempontból a kiterjesztés célja az, hogy **egy futáson belül** az indexdátumok mentén időben konzisztens státuszképzés történjen (dolgozat 5. fejezet: esetszintű `xi` küszöb és a Delta-mátrix/maszk konstrukció). Ez a tulajdonság nem garantálja, hogy **külön futtatások** (eltérő horizontkészletek) között a közös horizontokra kapott becslések változatlanok maradnak.

Gyakorlati értelmezés: a prefix-oszlopok közötti különbség lényegében azt jelenti, hogy a közös horizont becslése egy másik (ugyanazzal a maggal indított, de másképp felhasznált) véletlenszám-sorozaton alapul. Emiatt a különbségek nagyságrendje tipikusan összemérhető azzal a variabilitással, amit különböző seed-beállítások is okoznának.

További megállapítás a teljes notebook alapján: a seedek mentén képzett összevető táblák összesítése (az indexdátum–horizont párokhoz tartozó `|delta|` leíró statisztikái; lásd az "Összefoglalás (seed-variabilitás az eltérésben)" táblázatot, `summary_abs_delta`) azt mutatja, hogy a referencia és a kiterjesztett becslések közötti eltérések nagyságrendje a vizsgált beállítások mellett jellemzően a seed-választásból adódó Monte Carlo változékonysággal összemérhető. A különbségek előjele seed szerint is változhat, és az indexdátumok mentén nem rajzolódik ki tartós, egyirányú eltolódás; ez összhangban áll azzal, hogy a megfigyelt eltérések elsődleges forrása a futási szerkezet és a véletlenszám-felhasználás ütemezésének különbsége.

Mindezek alapján a kiterjesztett eljárás által adott becslések a referencia-megoldás eredményeivel **összevethetők**, és a megfigyelt eltérések a vizsgált beállítások mellett **módszertanilag elfogadható mértékűeknek** tekinthetők. Ennek megfelelően a kiterjesztett implementáció a referencia-eljárás logikájával **konzisztens** kimenetet szolgáltat, és a több indexdátum együttes kezelésére vonatkozó kiterjesztés validáltnak tekinthető a jelen vizsgálati keretben.


# todo
1. a kiértékelő táblákat erre a epsilon alapú különbségkre kell átvariálni
2. a konfidencia intervallumokat ugyanígy ki kell értékelni
3. Meg kell vizsgálni, hogy konvergál-e a nullához, ha növelem az ismétlésszámot
4. Ennek fényében átírni a megállapításokat